In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:
import sys
import urllib.request
import json
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def naver_shop(product: str) -> list[dict]:
    '''네이버 쇼핑의 최저가를 검색해서 판매하는곳 링크를 리스트로 10개 알려준다.'''
    client_id = 'Ff5Gg338LgLSGheX23Ne'
    client_secret = 'p2b2tBaYeg'
    encText = urllib.parse.quote(product)
    url = "https://openapi.naver.com/v1/search/shop?query=" + encText + "&sort=asc" # JSON 결과
    # url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    
    shop_result_list = []
    
    if(rescode==200):
        response_body = response.read()
        result_json = json.loads(response_body.decode('utf-8'))
        
        for temp in result_json['items']:
            dict_data = {}
            dict_data['물품명'] = temp['title']
            dict_data['판매링크'] = temp['link']
            
            shop_result_list.append(dict_data)
    else:
        print("Error Code:" + rescode)

    return shop_result_list


agent = create_agent(
    model='openai:gpt-5.4-mini',
    tools=[naver_shop],
    system_prompt= '''
    너는 네이버쇼핑의 최저가를 알려주는 에이전트야. 
    사용자의 질문에 상품명을 가지고 네이버 쇼핑의 최저가를 검색해서,
    판매물건의 이름과 판매물건의 주소를 알려줘.
    tool 결과를 바탕으로만 답변하고 추측해서 답변하지 않도록 주의해줘.
    '''
)

result = agent.invoke(
    {
        'messages': [{'role': 'user', 'content': '샤오미 가습기의 최저가정보를 알려줘'}]
    }
)

In [9]:
result['messages'][-1].content

'샤오미 가습기 최저가 정보입니다.  \ntool 결과 기준으로 확인된 판매물건 이름과 판매물건 주소를 알려드립니다.\n\n1. **미지아 MJJSQ06DY 스마트살균가습기**  \n   - 링크: https://smartstore.naver.com/main/products/7953103040\n\n2. **샤오미 미지아 스마트미 대용량 5리터 가습기 프로 CJSJSQ04LX**  \n   - 링크: https://invoc.co.kr/product/detail.html?product_no=699&cate_no=48&display_group=1&cafe_mkt=naver_ks&mkt_in=Y&ghost_mall_id=naver&ref=naver_open\n\n3. **샤오미 미지아 4 대용량 가습기 가정용 사무실 미니 가습기**  \n   - 링크: https://www.11st.co.kr/connect/Gateway.tmall?method=Xsite&prdNo=8121284365&tid=1000000061\n\n4. **(호환상품) 샤오미 자연기화식 가습기 Pro 방진 그물망 호환 필터 액세서리**  \n   - 링크: https://smartstore.naver.com/main/products/7864664274\n\n5. **샤오미 미지아 순정 자연 가습기 2세대 CJSJSQ01XY 전용 교체 필터**  \n   - 링크: https://invoc.co.kr/product/detail.html?product_no=945&cate_no=28&display_group=1&cafe_mkt=naver_ks&mkt_in=Y&ghost_mall_id=naver&ref=naver_open\n\n원하시면 제가 이 중에서 **가습기 본체만** 다시 추려서 정리해드릴게요.'

In [22]:
import sys
import urllib.request
import json
import requests
from bs4 import BeautifulSoup
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def naver_news_links(query: str, num_thread: int = 10) -> list[dict]:
    '''{query}로 검색한 {num_thread}개의 네이버 최신 뉴스 날짜와 뉴스 제목, 뉴스 링크를 반환한다.'''
    client_id = 'Ff5Gg338LgLSGheX23Ne'
    client_secret = 'p2b2tBaYeg'
    encText = urllib.parse.quote(query)
    url = "https://openapi.naver.com/v1/search/news?query=" + encText + f"&display={num_thread}" + "&sort=sim" # JSON 결과
    # url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)
    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    
    result_data_list = []
    
    if(rescode==200):
        response_body = response.read()
        result_json = json.loads(response_body.decode('utf-8'))
        
        for temp in result_json['items']:
            dict_data = {}
            dict_data['뉴스날짜'] = temp['pubDate']
            dict_data['뉴스제목'] = temp['title'].replace("<b>", "").replace("</b>", "")
            dict_data['뉴스링크'] = temp['link']
            
            result_data_list.append(dict_data)
    else:
        print("Error Code:" + rescode)
    
    return result_data_list
        

@tool
def scrape_web_page(url: str) -> str:
    """주어진 URL(링크)에 접속하여 웹페이지의 본문 텍스트를 읽어옵니다. 뉴스를 요약하거나 내용을 파악할 때 사용하세요."""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        paragraphs = soup.find_all('p')
        
        article_text = ' '.join([p.get_text(strip=True) for p in paragraphs])
        
        # 모델의 컨텍스트 창 제한을 막기 위해 최대 3000자로 제한
        return article_text[:300]
        
    except Exception as e:
        return f"해당 링크를 읽어오지 못했습니다. 에러: {e}"


tools = [naver_news_links, scrape_web_page]

agent = create_agent(
    'openai:gpt-5.4-mini',
    tools=tools,
    system_prompt='''
    당신은 네이버뉴스를 요약해주는 어시스턴트입니다. 
    사용자의 질문에서 query를 찾아 뉴스 링크를 tool로 리스트업 한 뒤,
    각각 요약해서 사용자에게 뉴스 날짜, 뉴스 제목, 간단한 뉴스 요약을 알려주세요.
    tool 결과를 바탕으로만 답변하고 추측해서 답변하지 않도록 주의해주세요.
    '''
    
)

result = agent.invoke(
    {
        'messages': [
            {'role': 'user', 'content': '트랜스포머가 어떤 구조로 되어있는지 궁금해.'}
        ]
    }
)

result

{'messages': [HumanMessage(content='트랜스포머가 어떤 구조로 되어있는지 궁금해.', additional_kwargs={}, response_metadata={}, id='99631a04-6acb-48ab-9ebf-61d9e867b395'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 313, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DVXio4f8wLrFVabQOio1SsBuNI3gp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9a4c-02d8-7e72-af0b-d4b983143ffe-0', tool_calls=[{'name': 'naver_news_links', 'args': {'query': '트랜스포머 구조', 'num_thread': 5}, 'id': 'call_2XfRBH4nW3LpVQkZJ4CnYM7I', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'